### **Hamiltonian and Commutation Analysis**

We consider the Hamiltonian
$$
H = -g \frac{w_{0}}{2}X_0X_1 -g{\frac{w_{0}}{2}} Y_0Y_1 - g\frac{w_{1}}{2} X_1X_2 - g\frac{w_{1}}{2} Y_1Y_2 .
$$
The corresponding unitary time evolution is
$$
U(t) = e^{-itH}.
$$
If a Hamiltonian can be written as a sum of commuting parts, \( H = A + B \) with \( [A,B]=0 \), then the unitary evolution factorizes exactly as
$$
e^{-itH} = e^{-itA} e^{-itB}.
$$
If the terms do not commute, this exact factorization is not possible.

We group the Hamiltonian as
$$
H = aH_{01} + bH_{12},
$$
where
$$
H_{01} = X_0X_1 + Y_0Y_1, \quad
H_{12} = X_1X_2 + Y_1Y_2.
$$
and $a = -g\frac{w_{0}}{2}$ and $b = -g\frac{w_{1}}{2}$


Within each group, the terms commute:
$$
[X_0X_1, Y_0Y_1] = 0, \quad
[X_1X_2, Y_1Y_2] = 0.
$$
However, the two groups share qubit 1, and Pauli operators acting on the same qubit do not commute. Consequently,
$$
[H_{01}, H_{12}] \neq 0,
$$
and the full Hamiltonian contains non-commuting terms.

### **Trotterized Time Evolution**

Since the Hamiltonian terms do not commute, the exact unitary evolution cannot be written as a single product of exponentials. To approximate the evolution, we use the Suzuki–Trotter decomposition.

The first-order Trotter approximation is
$$
e^{-itH}
\approx
\left(
e^{-i\Delta t a H_{01}}
e^{-i\Delta t b H_{12}}
\right)^n,
\quad
\Delta t = \frac{t}{n}.
$$

The approximation improves as the number of Trotter steps $n$ increases and becomes exact in the limit $n \to \infty$.

### **Circuit Construction**

We'll use the same method that we have used in our original document to create circuit for both $H_{01}$ and $H_{12}$. We start from $-Z_{0}$ as our simple Hamiltonian and try to reach the desired unitaries using 
$$
H = U^{\dagger} H_{0} U
$$
In case of $H_{01}$, we have two unitaries and there corresponding gate decomposition follows 
- **STRING 1**: $X_{0}X_{1}$
$$
e^{ i \theta_{0}  X_{0}X_{1}} = U^{\dagger} (Z_{0}) U
$$
where $\theta_{0} = \frac{gw_{0}}{2} \Delta t$ and,
$$
U = e^{i \frac{\pi}{4} X_{0}} e^{i \frac{\pi}{4}Z_{0}X_{1}}
$$
with
$$
e^{i \frac{\pi}{4}Z_{0}X_{1}} = e^{i \frac{\pi}{4} X_{1}} e^{i \frac{\pi}{4}Z_{0}} e^{-i \frac{\pi}{4}}CNOT^{0 \to 1}
$$
Hence, we can write the full circuit as
$$
e^{- i \Delta t a X_{0}X_{1}}  = CNOT^{0 \to 1} S_{0} R_{0}^{x}\left(\frac{\pi}{2}\right) \times \left(U_{1}^{0}(\theta_{0})\right)^{\dagger} \times  R_{0}^{x}\left(-\frac{\pi}{2}\right)S_{0}^{\dagger} CNOT^{0 \to 1} \tag{1}
$$
- **STRING 2**:  $Y_{0}Y_{1}$
In this case, we'll need two extra rotations, hence we get
$$
U = e^{i \frac{\pi}{4} X_{0}} e^{i \frac{\pi}{4}Z_{0}X_{1}} e^{i \frac{\pi}{4} Z_{0}} e^{i \frac{\pi}{4} Z_{1}}
$$
and the corresponding circuit will be, 
$$
e^{i \theta_{0} Y_{0}Y_{1}} = S_{0}S_{1} CNOT^{0 \to 1} S_{0} R_{0}^{x}\left(\frac{\pi}{2}\right) \times \left(U_{1}^{0}(\theta_{0})\right)^{\dagger} \times R_{0}^{x}\left(-\frac{\pi}{2}\right) S_{0}^{\dagger} CNOT^{0 \to 1}  S_{1}^{\dagger} S_{0}^{\dagger} \tag{2}
$$

Similarly, In case of H_{12},
- **STRING 1**: $X_{1}X_{2}$
This is similar as for string 1 of $H_{01}$, we just need to replace $0 \to 1$, $1 \to 2$, and $\theta_{0} \to \theta_{1}$, where $\theta_{1} = \frac{gw_{1}}{2}\Delta t$,
$$
e^{i \theta_{1} X_{1}X_{2}} =  CNOT^{1 \to 2} S_{1} R_{1}^{x}\left(\frac{\pi}{2}\right) \times \left(U_{1}^{1}(\theta_{1})\right)^{\dagger} \times  R_{1}^{x}\left(-\frac{\pi}{2}\right)S_{1}^{\dagger} CNOT^{1 \to 2} \tag{3}
$$
- **STRING 2**: $Y_{1}Y_{2}$
This one is similar to string 2 of $H_{01}$, with same replacements as previous, hence

$$
e^{i \theta_{1} Y_{1}Y_{2}} = S_{1}S_{2} CNOT^{1 \to 2} S_{1} R_{1}^{x}\left(\frac{\pi}{2}\right) \times \left(U_{1}^{1}(\theta_{1})\right)^{\dagger} \times R_{1}^{x}\left(-\frac{\pi}{2}\right) S_{1}^{\dagger} CNOT^{1 \to 2}  S_{2}^{\dagger} S_{1}^{\dagger} \tag{4}
$$


We can now put these together to create circuit for the unitary evolution of our Hamiltonian.

In [1]:
import qiskit
from qiskit import QuantumCircuit, transpile, ClassicalRegister, QuantumRegister
from qiskit.visualization import plot_histogram,plot_distribution
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService
from qiskit_aer import Aer, AerSimulator
from qiskit.circuit.library import UGate
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt
import pylatexenc
import numpy as np
import math

#### **Trotterization and Running on real backend**

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(
token="MyecTEwFVnhiA01nNvaqhXmIz6Z72cmU56ovGaytiO7g", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
instance="crn:v1:bluemix:public:quantum-computing:us-east:a/6f5d2cd8cddd46e9b5634b2c700fb95a:b5b5f0a4-24fe-476c-977a-cbc7c761049a::", # Optional
overwrite = True 
)
service = QiskitRuntimeService()
# service.saved_accounts() # To chek the account is saved successfully
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_marrakesh'

In [3]:
def build_circuit(eps, troter_steps):
    # DEFINE VARIABLES
    hbar = 1
    w = 1 
    gup_coeff = 0.0001
    w0 = 1 - (2 * gup_coeff)
    w1 = math.sqrt(2) - (4 * gup_coeff)
    phi_1 = (w0 * eps) / 2
    phi_2 = (w1 * eps) / 2
    tau = (hbar * w) / (2 * troter_steps) 
    # FOR TROTTERIZATION
    dphi_1 = phi_1 / troter_steps
    dphi_2 = phi_2 / troter_steps
    q0 = QuantumRegister(1, "q0")
    q1 = QuantumRegister(1,"q1")
    q2 = QuantumRegister(1,"q2")
    
    # NORMALIZED PROBABILITIES FOR INITIAL STATE VECTORS
    alpha = 1/np.sqrt(1 + 2*gup_coeff**2)
    beta  = np.sqrt(2)*gup_coeff/np.sqrt(1 + 2*gup_coeff**2)
    creg = ClassicalRegister(3,name = "measurement")
    # INITIAL STATE PREPARATION
    qc = QuantumCircuit(q0,q1,q2, creg)
    theta_prep = 2*np.arcsin(beta)
    qc.x(0)                         # |100>
    qc.cry(theta_prep, 0, 2)        # amplitude rotation
    qc.cx(2,0)
    for _ in range(troter_steps):
        qc.s(0)
        qc.cx(0, 1)
        qc.rx(np.pi/2, 0)
        qc.p(2 * dphi_1, 0)  # U1^0†
        qc.rx(-np.pi/2, 0)
        qc.cx(0, 1)
        qc.sdg(0)

        # ----------------- Subcircuit 2 -----------------
        qc.s(1)
        qc.z(0)
        qc.cx(0, 1)
        qc.rx(np.pi/2, 0)
        qc.p(2 * dphi_1, 0)
        qc.rx(-np.pi/2, 0)
        qc.cx(0, 1)
        qc.z(0)
        qc.sdg(1)

        # ----------------- Subcircuit 3 -----------------
        qc.p(2 * tau, 1)

        qc.barrier()

        # -------------------BLOCK 2 ----------------------
        qc.s(1)
        qc.cx(1, 2)
        qc.rx(np.pi/2, 1)
        qc.p(2 * dphi_2, 1) 
        qc.rx(-np.pi/2, 1)
        qc.cx(1, 2)
        qc.sdg(1)

        # ----------------- Subcircuit 2 -----------------
        qc.s(2)
        qc.z(1)
        qc.cx(1, 2)
        qc.rx(np.pi/2, 1)
        qc.p(2 * dphi_2, 1)  # U1^0†
        qc.rx(-np.pi/2, 1)
        qc.cx(1, 2)
        qc.z(1)
        qc.sdg(2)

        # ----------------- Subcircuit 3 -----------------
        qc.p(2 * tau, 2)
    qc.measure([0,1,2],[0,1,2])
    return qc
all_circuits = []
metadata = []
# eps_list = [0.001,0.002,0.003,0.004,0.005,0.006] # math.pow(10,-3), math.pow(10,-5) 
eps_list =  np.linspace(0.15, 0.25, 25)
troter_steps = [20]
for steps in troter_steps:
    for eps in eps_list:

        qc = build_circuit(eps, steps)

        all_circuits.append(qc)
        metadata.append((steps, eps))

# -----------------------------------
# 2️⃣ Transpile ALL circuits together
# -----------------------------------
transpiled_circuits = transpile(all_circuits, backend, optimization_level=3)

# -----------------------------------
# 3️⃣ Submit ONE job
# -----------------------------------
sampler = Sampler(mode=backend)

job = sampler.run(transpiled_circuits, shots=1024)
print("Job ID:", job.job_id())
print("Backend:", backend.name)
print("Status:", job.status())
job_result = job.result()

# -----------------------------------
# 4️⃣ Organize results nicely
# -----------------------------------
results = {}

for i, (steps, eps) in enumerate(metadata):

    counts = job_result[i].data.measurement.get_counts()

    if steps not in results:
        results[steps] = {}

    results[steps][eps] = counts

# -----------------------------------
# 5️⃣ Print clean output
# -----------------------------------
for steps in results:
    print("\n==============================")
    print("Trotter steps =", steps)
    print("==============================")

    for eps in results[steps]:
        print(f"eps = {eps} → {results[steps][eps]}")

Job ID: d7jg7chs7cos73ejs3mg
Backend: ibm_marrakesh
Status: RUNNING

Trotter steps = 20
eps = 0.15 → {'001': 727, '101': 45, '000': 81, '010': 45, '111': 24, '011': 70, '100': 17, '110': 15}
eps = 0.15416666666666667 → {'010': 29, '001': 771, '101': 44, '000': 87, '011': 52, '111': 27, '100': 4, '110': 10}
eps = 0.15833333333333333 → {'001': 790, '100': 16, '110': 9, '011': 31, '101': 58, '010': 20, '111': 29, '000': 71}
eps = 0.1625 → {'001': 712, '011': 61, '010': 73, '000': 87, '101': 40, '111': 29, '110': 11, '100': 11}
eps = 0.16666666666666666 → {'000': 104, '011': 37, '001': 723, '100': 12, '101': 51, '010': 56, '110': 17, '111': 24}
eps = 0.17083333333333334 → {'001': 716, '101': 81, '000': 95, '111': 33, '010': 27, '011': 48, '100': 14, '110': 10}
eps = 0.175 → {'001': 772, '000': 79, '010': 35, '101': 31, '011': 71, '111': 18, '110': 9, '100': 9}
eps = 0.17916666666666667 → {'001': 728, '011': 60, '111': 29, '110': 16, '010': 71, '100': 9, '101': 42, '000': 69}
eps = 0.183333

In [4]:
def counts_to_probs(counts_dict):
    shots = sum(counts_dict.values())
    return {k: v / shots for k, v in counts_dict.items()}

allowed = ['010','100','001']

postselected_results = {}

for steps in results:
    postselected_results[steps] = {}

    for eps in results[steps]:
        counts = results[steps][eps]
        probs = counts_to_probs(counts)

        total = sum(probs.get(k, 0.0) for k in allowed)

        if total == 0:
            post_probs = {k: 0 for k in allowed}
        else:
            post_probs = {k: probs.get(k, 0.0) / total for k in allowed}

        postselected_results[steps][eps] = post_probs

        print(f"steps={steps}, eps={eps} → {post_probs}")
# def counts_to_probs(counts_dict):
#     shots = sum(counts_dict.values())
#     return {k: v / shots for k, v in counts_dict.items()}

# allowed = ['010','100', '001'] 

# # Convert counts → probabilities
# probs = counts_to_probs(results[steps][eps])

# # Compute total probability inside allowed subspace
# total = sum(probs.get(k, 0.0) for k in allowed)

# # Avoid division by zero
# if total == 0:
#     print("No allowed states measured!")
#     postselected_probs = {k: 0 for k in allowed}
# else:
#     postselected_probs = {k: probs.get(k, 0.0) / total for k in allowed}

# print(postselected_probs)

steps=20, eps=0.15 → {'010': 0.057034220532319393, '100': 0.021546261089987327, '001': 0.9214195183776933}
steps=20, eps=0.15416666666666667 → {'010': 0.036069651741293535, '100': 0.004975124378109453, '001': 0.9589552238805971}
steps=20, eps=0.15833333333333333 → {'010': 0.024213075060532687, '100': 0.01937046004842615, '001': 0.9564164648910412}
steps=20, eps=0.1625 → {'010': 0.09170854271356783, '100': 0.013819095477386936, '001': 0.8944723618090452}
steps=20, eps=0.16666666666666666 → {'010': 0.07079646017699115, '100': 0.015170670037926675, '001': 0.9140328697850821}
steps=20, eps=0.17083333333333334 → {'010': 0.035667107001321, '100': 0.018494055482166448, '001': 0.9458388375165125}
steps=20, eps=0.175 → {'010': 0.0428921568627451, '100': 0.011029411764705883, '001': 0.946078431372549}
steps=20, eps=0.17916666666666667 → {'010': 0.08787128712871287, '100': 0.011138613861386138, '001': 0.900990099009901}
steps=20, eps=0.18333333333333332 → {'010': 0.034610630407911, '100': 0.01730

In [5]:
def compute_true_probabilities(eps_list, trotter_steps):
    true_results = {}

    for steps in trotter_steps:
        true_results[steps] = {}

        for eps in eps_list:

            # Build circuit
            qc = build_circuit(eps, steps)

            # Remove measurements (important!)
            qc_no_meas = qc.remove_final_measurements(inplace=False)

            # Get exact statevector
            state = Statevector.from_instruction(qc_no_meas)

            # Convert to probabilities
            probs = state.probabilities_dict()

            true_results[steps][eps] = probs

    return true_results
true_probs = compute_true_probabilities(eps_list, troter_steps)
print(true_probs)

{20: {np.float64(0.15): {np.str_('001'): np.float64(0.9795383670706228), np.str_('010'): np.float64(0.020205991603855358), np.str_('100'): np.float64(0.0002556413255475845), np.str_('111'): np.float64(5.007417855406814e-32)}, np.float64(0.15416666666666667): {np.str_('001'): np.float64(0.9783990690117639), np.str_('010'): np.float64(0.021315612614101483), np.str_('100'): np.float64(0.00028531837415639907), np.str_('111'): np.float64(9.321500930834218e-32)}, np.float64(0.15833333333333333): {np.str_('001'): np.float64(0.9772300173638567), np.str_('010'): np.float64(0.02245249976997367), np.str_('100'): np.float64(0.00031748286618947746), np.str_('111'): np.float64(2.2263750157116446e-31)}, np.float64(0.1625): {np.str_('001'): np.float64(0.976031328674329), np.str_('010'): np.float64(0.023616404457467762), np.str_('100'): np.float64(0.0003522668682279198), np.str_('111'): np.float64(2.565338685923799e-31)}, np.float64(0.16666666666666666): {np.str_('001'): np.float64(0.9748031223558896),

In [6]:
def classical_fidelity(p_true, p_exp):
    all_keys = set(p_true.keys()).union(set(p_exp.keys()))
    fidelity_sum = 0.0

    for key in all_keys:
        pt = p_true.get(key, 0.0)
        pe = p_exp.get(key, 0.0)
        fidelity_sum += np.sqrt(pt * pe)

    return fidelity_sum**2

fidelity_results = {}

for steps in results:
    fidelity_results[steps] = {}

    for eps in results[steps]:

        # Exact probabilities from statevector
        p_true = true_probs[steps][eps]

        # Hardware probabilities → apply postselection
        counts = results[steps][eps]
        p_raw = counts_to_probs(counts)

        # Apply postselection: keep only keys present in ideal probs
        allowed_keys = p_true.keys()
        total = sum(p_raw[k] for k in allowed_keys if k in p_raw)
        p_exp = {k: p_raw.get(k, 0.0)/total for k in allowed_keys}

        # Compute classical fidelity using postselected probs
        F = classical_fidelity(p_true, p_exp)

        fidelity_results[steps][eps] = F

        print(f"Trotter steps = {steps}, eps = {eps} → Fidelity = {100 * F:.6f}")

Trotter steps = 20, eps = 0.15 → Fidelity = 94.412633
Trotter steps = 20, eps = 0.15416666666666667 → Fidelity = 96.276993
Trotter steps = 20, eps = 0.15833333333333333 → Fidelity = 95.176635
Trotter steps = 20, eps = 0.1625 → Fidelity = 93.252577
Trotter steps = 20, eps = 0.16666666666666666 → Fidelity = 94.791373
Trotter steps = 20, eps = 0.17083333333333334 → Fidelity = 94.463031
Trotter steps = 20, eps = 0.175 → Fidelity = 96.975584
Trotter steps = 20, eps = 0.17916666666666667 → Fidelity = 94.193368
Trotter steps = 20, eps = 0.18333333333333332 → Fidelity = 95.737618
Trotter steps = 20, eps = 0.1875 → Fidelity = 96.159307
Trotter steps = 20, eps = 0.19166666666666665 → Fidelity = 95.791036
Trotter steps = 20, eps = 0.19583333333333333 → Fidelity = 96.034533
Trotter steps = 20, eps = 0.2 → Fidelity = 96.136043
Trotter steps = 20, eps = 0.20416666666666666 → Fidelity = 95.499081
Trotter steps = 20, eps = 0.20833333333333331 → Fidelity = 96.685295
Trotter steps = 20, eps = 0.2125 → F

In [ ]:
# -----------------------------------
# Filter eps values by fidelity
# -----------------------------------

FIDELITY_THRESHOLD = 0.95   # 95%

high_fidelity_eps = []

# Since you only have one Trotter step, access it directly
steps = troter_steps[0]  # 5 in your case

for eps, F in fidelity_results[steps].items():
    if F >= FIDELITY_THRESHOLD:
        high_fidelity_eps.append((eps, F))

# Sort by fidelity (highest first)
high_fidelity_eps.sort(key=lambda x: x[1], reverse=True)

# Print results
print(f"\nEpsilon values with fidelity ≥ {100*FIDELITY_THRESHOLD:.0f}% for {steps} Trotter steps:")
for eps, F in high_fidelity_eps:
    print(f"eps = {eps:.6f}, Fidelity = {100*F:.2f}%")


Epsilon values with fidelity ≥ 95% for 20 Trotter steps:
eps = 0.216667, Fidelity = 97.05%
eps = 0.175000, Fidelity = 96.98%
eps = 0.225000, Fidelity = 96.84%
eps = 0.208333, Fidelity = 96.69%
eps = 0.237500, Fidelity = 96.46%
eps = 0.233333, Fidelity = 96.36%
eps = 0.212500, Fidelity = 96.29%
eps = 0.154167, Fidelity = 96.28%
eps = 0.187500, Fidelity = 96.16%
eps = 0.200000, Fidelity = 96.14%
eps = 0.220833, Fidelity = 96.06%
eps = 0.195833, Fidelity = 96.03%
eps = 0.250000, Fidelity = 95.88%
eps = 0.245833, Fidelity = 95.81%
eps = 0.191667, Fidelity = 95.79%
eps = 0.183333, Fidelity = 95.74%
eps = 0.229167, Fidelity = 95.68%
eps = 0.204167, Fidelity = 95.50%
eps = 0.158333, Fidelity = 95.18%


#### **Calculate Analytical Probabilities**

In [9]:
gup_coeff = 0.0001
for eps in eps_list:

    theta = eps * np.sqrt(3)

    C0 = (2 + np.cos(theta))/3 \
         + gup_coeff*((14/9)*(np.cos(theta) - 1) + (10/9)*theta*np.sin(theta))

    C1 = 1j/np.sqrt(3) * ( np.sin(theta) \
         + (10/3)*gup_coeff*(np.sin(theta) - theta*np.cos(theta)) )

    C2 = np.sqrt(2)/3*(np.cos(theta) - 1) \
         + (gup_coeff*np.sqrt(2)/9)*(1 + 8*np.cos(theta) + 10*theta*np.sin(theta))

    print(f"eps = {eps}")
    # AMPLITUDE^{2} = PROBABILITY
    print("p0 =", abs(C0**2)) 
    print("p1 =", abs(C1**2))
    print("p2 =", abs(C2**2))
    print()

eps = 0.15
p0 = 0.977755765570044
p1 = 0.021998615818142983
p2 = 0.0002456404302783142

eps = 0.15416666666666667
p0 = 0.9765176549912085
p1 = 0.023208186232922177
p2 = 0.00027418069785643747

eps = 0.15833333333333333
p0 = 0.9752472599038369
p1 = 0.024447646535883955
p2 = 0.00030511558874431037

eps = 0.1625
p0 = 0.9739447109579407
p1 = 0.02571673868275398
p2 = 0.0003385724972157235

eps = 0.16666666666666666
p0 = 0.97261014201976
p1 = 0.02701519846017953
p2 = 0.00037468177039239053

eps = 0.17083333333333334
p0 = 0.9712436901510436
p1 = 0.028342755540736207
p2 = 0.00041357667395998496

eps = 0.175
p0 = 0.9698454955878412
p1 = 0.029699133539207685
p2 = 0.0004553933570959104

eps = 0.17916666666666667
p0 = 0.9684157017188141
p1 = 0.031084050070126566
p2 = 0.0005002708166164974

eps = 0.18333333333333332
p0 = 0.9669544550630725
p1 = 0.03249721680656443
p2 = 0.0005483508603514506

eps = 0.1875
p0 = 0.9654619052475362
p1 = 0.0339383395401588
p2 = 0.0005997780697536174

eps = 0.19166666666